In [18]:
import mujoco
mjcf_path = "../models/ur10/ur10.xml"
urdf_path = "../models/ur10/ur10.urdf"
mj_model = mujoco.MjModel.from_xml_path(urdf_path)

# 2. Zapisujemy go jako natywny plik MuJoCo (MJCF)
mujoco.mj_saveLastXML(mjcf_path, mj_model)

In [11]:
import crocoddyl
import pinocchio
import numpy as np

class DifferentialFreeFwdDynamicsModelDerived(
    crocoddyl.DifferentialActionModelAbstract
):
    def __init__(self, state, actuationModel, costModel):
        crocoddyl.DifferentialActionModelAbstract.__init__(
            self, state, actuationModel.nu, costModel.nr
        )
        self.actuation = actuationModel
        self.costs = costModel
        self.enable_force = True
        self.armature = np.matrix(np.zeros(0))

    def calc(self, data, x, u=None):
        if u is None:
            q, v = x[: self.state.nq], x[-self.state.nv :]
            pinocchio.computeAllTerms(self.state.pinocchio, data.pinocchio, q, v)
            self.costs.calc(data.costs, x)
            data.cost = data.costs.cost
        else:
            q, v = x[: self.state.nq], x[-self.state.nv :]
            self.actuation.calc(data.actuation, x, u)
            tau = data.actuation.tau
            # Computing the dynamics using ABA or manually for armature case
            if self.enable_force:
                data.xout[:] = pinocchio.aba(
                    self.state.pinocchio, data.pinocchio, q, v, tau
                )
            else:
                pinocchio.computeAllTerms(self.state.pinocchio, data.pinocchio, q, v)
                data.M = data.pinocchio.M
                if self.armature.size == self.state.nv:
                    data.M[range(self.state.nv), range(self.state.nv)] += self.armature
                data.Minv = np.linalg.inv(data.M)
                data.xout[:] = np.dot(data.Minv, (tau - data.pinocchio.nle))
            # Computing the cost value and residuals
            pinocchio.forwardKinematics(self.state.pinocchio, data.pinocchio, q, v)
            pinocchio.updateFramePlacements(self.state.pinocchio, data.pinocchio)
            self.costs.calc(data.costs, x, u)
            data.cost = data.costs.cost

    def calcDiff(self, data, x, u=None):
        if u is None:
            self.costs.calcDiff(data.costs, x)
        else:
            nq, nv = self.state.nq, self.state.nv
            q, v = x[:nq], x[-nv:]
            # Computing the actuation derivatives
            self.actuation.calcDiff(data.actuation, x, u)
            tau = data.actuation.tau
            # Computing the dynamics derivatives
            if self.enable_force:
                pinocchio.computeABADerivatives(
                    self.state.pinocchio, data.pinocchio, q, v, tau
                )
                ddq_dq = data.pinocchio.ddq_dq
                ddq_dv = data.pinocchio.ddq_dv
                data.Fx[:, :] = np.hstack([ddq_dq, ddq_dv]) + np.dot(
                    data.pinocchio.Minv, data.actuation.dtau_dx
                )
                data.Fu[:, :] = np.dot(data.pinocchio.Minv, data.actuation.dtau_du)
            else:
                pinocchio.computeRNEADerivatives(
                    self.state.pinocchio, data.pinocchio, q, v, data.xout
                )
                ddq_dq = np.dot(
                    data.Minv, (data.actuation.dtau_dx[:, :nv] - data.pinocchio.dtau_dq)
                )
                ddq_dv = np.dot(
                    data.Minv, (data.actuation.dtau_dx[:, nv:] - data.pinocchio.dtau_dv)
                )
                data.Fx[:, :] = np.hstack([ddq_dq, ddq_dv])
                data.Fu[:, :] = np.dot(data.Minv, data.actuation.dtau_du)
            # Computing the cost derivatives
            self.costs.calcDiff(data.costs, x, u)

    def createData(self):
        data = DifferentialFreeFwdDynamicsDataDerived(self)
        return data

    def set_armature(self, armature):
        if armature.size is not self.state.nv:
            print("The armature dimension is wrong, we cannot set it.")
        else:
            self.enable_force = False
            self.armature = armature.T


class DifferentialFreeFwdDynamicsDataDerived(crocoddyl.DifferentialActionDataAbstract):
    def __init__(self, model):
        crocoddyl.DifferentialActionDataAbstract.__init__(self, model)
        self.pinocchio = pinocchio.Model.createData(model.state.pinocchio)
        self.multibody = crocoddyl.DataCollectorMultibody(self.pinocchio)
        self.actuation = model.actuation.createData()
        self.costs = model.costs.createData(self.multibody)
        self.costs.shareMemory(self)
        self.Minv = None

In [4]:
import time

import numpy as np
import mujoco
import mujoco.viewer
import pinocchio as pin
import optimal.PDController

mjcf_path = "../models/ur10/ur10.xml"
urdf_path = "../models/ur10/ur10.urdf"
robot_mj_model = mujoco.MjModel.from_xml_path(mjcf_path)
robot_pin_model = pin.buildModelFromUrdf(urdf_path)
# print(type(robot_pin_model))
# print(robot_pin_model.nq)
# print(robot_pin_model.nv)
# q_ref = pin.neutral(robot_pin_model)
# reduced_model = pin.buildReducedModel(robot_pin_model, [1], q_ref)
# print(type(reduced_model))
# print(robot_pin_model.nq)
# print(robot_pin_model.nv)

q0 = np.array([0.0, -np.pi / 4, 0.0, -np.pi / 2, 0.0, np.pi / 3])
x0 = np.concatenate([q0, np.zeros(robot_pin_model.nv)])
# Create the cost functions

target = np.array([0.4, 0.0, 0.4])
target = np.array([0.6, 0.2, 0.0])

q_target = np.array([-2.534333, -3.794926, 2.220738, -1.761211, -0.306910, 1.047198])
x_target = np.concatenate([q_target, np.zeros(robot_pin_model.nv)])


controller = optimal.PDController.PDController(robot_pin_model, kp=1.0, kd=0.2, torque_limits=robot_mj_model.actuator_ctrlrange[:, 1])
xs, us = controller.compute_control(x0, x_target, horizon=2000, control_dt=robot_mj_model.opt.timestep)

robot_mj_data = mujoco.MjData(robot_mj_model)
if True:
    with mujoco.viewer.launch_passive(robot_mj_model, robot_mj_data) as viewer:
        for u_cmd in us:
            print(u_cmd)
            robot_mj_data.ctrl[:] = u_cmd
            mujoco.mj_step(robot_mj_model, robot_mj_data)
            viewer.sync()
            time.sleep(robot_mj_model.opt.timestep)
        input("Press Enter to continue...")
        

[-2.53433300e+00 -3.00952784e+00  2.22073800e+00 -1.90414673e-01
 -3.06910000e-01  4.48803402e-07]
[-2.53471173 -3.01341145  2.22287072 -0.17461923 -0.26910591 -0.01404464]
[-2.53509324 -3.01733604  2.22506993 -0.16017804 -0.23573452 -0.01007214]
[-2.53547747 -3.02130114  2.22730719 -0.14653871 -0.20595972 -0.00985034]
[-2.53586466 -3.02530704  2.22958754 -0.13370622 -0.1794817  -0.00896902]
[-2.53625488 -3.02935387  2.23190932 -0.12158376 -0.15594111 -0.0083095 ]
[-2.53664815 -3.0334418   2.23427236 -0.11010483 -0.13503442 -0.00768473]
[-2.53704446 -3.03757102  2.23667629 -0.09920555 -0.11648596 -0.00712451]
[-2.53744372 -3.04174172  2.23912086 -0.08882932 -0.10004954 -0.00661563]
[-2.53784585 -3.0459541   2.24160589 -0.07892509 -0.08550441 -0.00615431]
[-2.53825076 -3.05020836  2.24413125 -0.06944701 -0.07265283 -0.00573553]
[-2.53865831 -3.05450474  2.24669689 -0.06035379 -0.06131761 -0.00535512]
[-2.5390684  -3.05884346  2.24930281 -0.05160832 -0.05133998 -0.00500927]
[-2.53948088 

In [36]:
q0 = np.array([0.0, -np.pi / 4, 0.0, -np.pi / 2, 0.0, np.pi / 3])
x0 = np.concatenate([q0, np.zeros(robot_pin_model.nv)])

# Create the cost functions
target = np.array([0.4, 0.0, 0.4])
target = np.array([0.6, 0.2, 0.0])
state = crocoddyl.StateMultibody(robot_pin_model)
frameTranslationResidual = crocoddyl.ResidualModelFrameTranslation(
    state, robot_pin_model.getFrameId("wrist_3_link"), target
)
goalTrackingCost = crocoddyl.CostModelResidual(state, frameTranslationResidual)
xRegCost = crocoddyl.CostModelResidual(state, crocoddyl.ResidualModelState(state))
uRegCost = crocoddyl.CostModelResidual(state, crocoddyl.ResidualModelControl(state))

# Create cost model per each action model
runningCostModel = crocoddyl.CostModelSum(state)
terminalCostModel = crocoddyl.CostModelSum(state)

# Then let's added the running and terminal cost functions
runningCostModel.addCost("gripperPose", goalTrackingCost, 1e2)
runningCostModel.addCost("stateReg", xRegCost, 1e-4)
runningCostModel.addCost("ctrlReg", uRegCost, 1e-7)
terminalCostModel.addCost("gripperPose", goalTrackingCost, 1e5)
terminalCostModel.addCost("stateReg", xRegCost, 1e-4)
terminalCostModel.addCost("ctrlReg", uRegCost, 1e-7)

# Running and terminal action models
DT = robot_mj_model.opt.timestep
actuationModel = crocoddyl.ActuationModelFull(state)
runningModel = crocoddyl.IntegratedActionModelEuler(
    crocoddyl.DifferentialActionModelFreeFwdDynamics(
        state, actuationModel, runningCostModel
    ),
    DT,
)
terminalModel = crocoddyl.IntegratedActionModelEuler(
    crocoddyl.DifferentialActionModelFreeFwdDynamics(
        state, actuationModel, terminalCostModel
    ),
    0.0,
)

In [45]:
# For this optimal control problem, we define 250 knots (or running action
# models) plus a terminal knot
T = 3000
print(len(x0[:6]))
print(x0)
u0 = np.zeros(actuationModel.nu * T)
robot_mj_data = mujoco.MjData(robot_mj_model)
print(robot_mj_data.xpos[mujoco.mj_name2id(robot_mj_model, mujoco.mjtObj.mjOBJ_BODY, "wrist_3_link")])
# compute xyz position of wrist_3_link using joint positions and forward kinematics
robot_data = robot_pin_model.createData()
pinocchio.forwardKinematics(robot_pin_model, robot_data, q0)
pinocchio.updateFramePlacements(robot_pin_model, robot_data)
frame_id = robot_pin_model.getFrameId("wrist_3_link")
wrist_pos = robot_data.oMf[frame_id].translation
print("wrist_3_link position (Pinocchio FK):", wrist_pos)

problem = crocoddyl.ShootingProblem(x0, [runningModel] * T, terminalModel)

# Creating the DDP solver for this OC problem, defining a logger
solver = crocoddyl.SolverFDDP(problem)
solver.solve()
log = crocoddyl.CallbackLogger()
pinocchio.forwardKinematics(robot_pin_model, robot_data, robot_mj_data.qpos)

print(robot_mj_data.ctrl)
with mujoco.viewer.launch_passive(robot_mj_model, robot_mj_data) as viewer:
    for u_cmd in solver.us:
        print("u_cmd:", u_cmd)
        robot_mj_data.ctrl[:] = u_cmd
        mujoco.mj_step(robot_mj_model, robot_mj_data)
        viewer.sync()
        time.sleep(robot_mj_model.opt.timestep)
    input("Press Enter to continue...")

# print position of wrist_3_link
print(robot_mj_data.xpos[mujoco.mj_name2id(robot_mj_model, mujoco.mjtObj.mjOBJ_BODY, "wrist_3_link")])

pinocchio.forwardKinematics(robot_pin_model, robot_data, robot_mj_data.qpos)
pinocchio.updateFramePlacements(robot_pin_model, robot_data)
frame_id = robot_pin_model.getFrameId("wrist_3_link")
wrist_pos = robot_data.oMf[frame_id].translation
print("wrist_3_link position (Pinocchio FK):", wrist_pos)

6
[ 0.         -0.78539816  0.         -1.57079633  0.          1.04719755
  0.          0.          0.          0.          0.          0.        ]
[0. 0. 0.]
wrist_3_link position (Pinocchio FK): [0.91923882 0.256141   1.04653882]


ArgumentError: Python argument types in
    SolverFDDP.solve(SolverFDDP, list)
did not match C++ signature:
    solve(crocoddyl::SolverFDDP {lvalue} self)
    solve(crocoddyl::SolverFDDP {lvalue} self, std::vector<Eigen::Matrix<double, -1, 1, 0, -1, 1>, std::allocator<Eigen::Matrix<double, -1, 1, 0, -1, 1> > > init_xs)
    solve(crocoddyl::SolverFDDP {lvalue} self, std::vector<Eigen::Matrix<double, -1, 1, 0, -1, 1>, std::allocator<Eigen::Matrix<double, -1, 1, 0, -1, 1> > > init_xs, std::vector<Eigen::Matrix<double, -1, 1, 0, -1, 1>, std::allocator<Eigen::Matrix<double, -1, 1, 0, -1, 1> > > init_us)
    solve(crocoddyl::SolverFDDP {lvalue} self, std::vector<Eigen::Matrix<double, -1, 1, 0, -1, 1>, std::allocator<Eigen::Matrix<double, -1, 1, 0, -1, 1> > > init_xs, std::vector<Eigen::Matrix<double, -1, 1, 0, -1, 1>, std::allocator<Eigen::Matrix<double, -1, 1, 0, -1, 1> > > init_us, unsigned long maxiter)
    solve(crocoddyl::SolverFDDP {lvalue} self, std::vector<Eigen::Matrix<double, -1, 1

In [53]:
import mujoco, mujoco.viewer
model = mujoco.MjModel.from_xml_path('../models/ur5e/ur5e.urdf')
data = mujoco.MjData(model)
with mujoco.viewer.launch_passive(model, data) as viewer:
    mujoco.mj_step(model, data)
    viewer.sync()
    input("Press Enter to continue...")
mujoco.mj_saveLastXML('../models/robot.xml', model)


ValueError: Error: Error opening file 'wrist1.stl'

In [55]:
from pathlib import Path
import mujoco, mujoco.viewer

urdf = (Path.cwd() / "models" / "ur5e" / "ur5e.urdf").resolve()
if not urdf.exists():
    urdf = (Path.cwd().parent / "models" / "ur5e" / "ur5e.urdf").resolve()

model = mujoco.MjModel.from_xml_path(str(urdf))
data = mujoco.MjData(model)

with mujoco.viewer.launch_passive(model, data) as viewer:
    mujoco.mj_step(model, data)
    viewer.sync()
    input("Press Enter to continue...")

mujoco.mj_saveLastXML(str(urdf.with_suffix(".xml")), model)


In [61]:
import mujoco
model = mujoco.MjModel.from_xml_path('../models/combined.xml')
data = mujoco.MjData(model)
with mujoco.viewer.launch_passive(model, data) as viewer:
    for _ in range(10):
        mujoco.mj_step(model, data)
        viewer.sync()
    input("Press Enter to continue...")


In [7]:
import pinocchio
import mujoco

# 1. Tworzenie MÓZGU (Crocoddyl / Pinocchio)
# Ładujemy czysty model ramienia w formacie URDF
urdf_path = "../models/ur5e/ur5e.urdf"
rmodel = pinocchio.buildModelFromUrdf(urdf_path)

# 2. Tworzenie ŚWIATA (MuJoCo)
# Opcja A: Jeśli symulujesz samo ramię, ładujesz ten sam URDF prosto do MuJoCo
mj_model = mujoco.MjModel.from_xml_path(urdf_path)

# 2. Zapisujemy go jako natywny plik MuJoCo (MJCF)
mujoco.mj_saveLastXML("../models/ur5e/ur5e.xml", mj_model)

print("Konwersja zakończona! Wygenerowano ur5e.xml")

# Opcja B: Jeśli masz złożoną scenę (podłoga, kamery, klocek, stół), 
# ładujesz plik scene.xml, który W SOBIE zawiera (include) plik URDF.
mj_model = mujoco.MjModel.from_xml_path("../models/test_scene.xml")

Konwersja zakończona! Wygenerowano ur5e.xml


ValueError: Error: Error opening file 'mesh/models/ur5e/collision/base.stl'